In [ ]:
import pandas as pd
import seaborn as sns
import joblib

from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split,GridSearchCV
from sklearn.metrics import r2_score,mean_absolute_error

In [ ]:
df = pd.read_csv("Bengaluru_House_Data.csv")

In [ ]:
#checking and removing unwanted columns- area_type, availability, society, balcony
df= df.drop(['area_type', 'balcony', 'society', 'availability'], axis=1) 
df.head()
              

In [ ]:
df.isnull().sum()

In [ ]:
df["location"].value_counts()

In [ ]:
df["location"]= df["location"].fillna("sarjapur road")

In [ ]:
df["location"].value_counts()

In [ ]:
df.info()

In [ ]:
df

In [ ]:
df['bath'].value_counts()

In [ ]:
df['bath'] = df['bath'].fillna(df['bath'].median())

In [ ]:
df['bath'].value_counts()

In [ ]:
df['bath'].isnull().sum()

In [ ]:
df['bath'].isnull().sum()

In [ ]:
df['bath']=df['bath'].astype(int)
df['bath'].unique()

In [ ]:
df.drop_duplicates()


In [ ]:
df.value_counts()

In [ ]:
df['location'].value_counts()

In [ ]:
df['location']=df['location'].apply(lambda x:x.strip()) #.apply()is used to apply a function along an axis of the DataFrame. In this case, we are applying the strip() function to remove any leading or trailing whitespace from the location values.

loc = df['location'].value_counts()#storing the values of the column.
loc_1than_10 = loc[loc<=10] #storing the values which are less than 10 in a variable.

df['location'] = df['location'].apply(lambda x: 'other' if x in loc_1than_10 else x) #replacing the values which are less than 10 with other.
df['location'].value_counts()

In [ ]:
df['size'].value_counts()#has some null values and also has some values like 2 BHK, 3 BHK etc. we will handle it in the next step.

out = [int(i.split()[0]) if isinstance(i, str) else 0 for i in df['size']]

df['bhk'] = out
df

In [ ]:
print(out)

In [ ]:
def clean_sqft (sqft):
    li=sqft.split('-')
    try:
        if len(li)==2:
            return (float(li[0]) + float(li[0]))/2
        else:
            return float(li[0])
        
    except:
        return None
    

df['total_sqft'] = df['total_sqft'].apply(clean_sqft)

df['total_sqft'] = df['total_sqft'].fillna(round(df['total_sqft'].mean()))

In [ ]:
df['total_sqft'].value_counts()

In [ ]:
df['price_per_sqft'] = df['price']*100000/df['total_sqft']#price is in lakhs and total_sqft is in sqft so we are multiplying price by 100000 to convert it into rupees.
df

In [ ]:
#bcz no room exist with 1 sqft but total_sqft column has some values i.e 1 sqft which is an outlier
#so we r filtering the data which is greater than or equal to 300 sqft

df = df[df['total_sqft']/df['bhk']>=300]

In [ ]:
df.describe()

In [ ]:
df = df[df['bhk']<6]

In [ ]:
df = df[df["bath"]<df["bhk"]+2]

In [ ]:
sns.boxplot(x="price_per_sqft",data=df)

In [ ]:
q1 = df["price_per_sqft"].quantile(0.25)
q3 = df["price_per_sqft"].quantile(0.75)

IQR = q3-q1

lower = q1-0.5*IQR
upper = q3+0.5*IQR

df = df[(df["price_per_sqft"]>=lower)&(df["price_per_sqft"]<=upper)]
sns.boxplot(x="price_per_sqft",data = df)

In [ ]:
df.reset_index(inplace=True)
df

In [ ]:
df = df.drop(["index","size"], axis=1)
df

In [ ]:
df = pd.get_dummies(df,columns=['location'],drop_first=True,dtype=int)

In [ ]:
df.columns

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df)

In [ ]:
X = df.drop(["price",'price_per_sqft'],axis=1)
y=df.price

In [ ]:
#split the data into training and testing data
Xtrain,Xtest,ytrain,ytest = train_test_split(X,y,test_size=0.3,random_state=42)

In [ ]:
model = RandomForestRegressor(random_state=42)
params = {
    "n_estimators":[100,150,200],
    "max_depth":[3,4,5,6,7]
}

grid = GridSearchCV(estimator=model,param_grid=params,cv=5)

grid.fit(Xtrain,ytrain)

print("Best params:",grid.best_params_)
print("Best  Score :",grid.best_score_)

In [ ]:
best_model = grid.best_estimator_

In [ ]:
ypred = grid.predict(Xtest)
ypred

In [ ]:
print("Training Eff: ",grid.score(Xtrain,ytrain))
print("Testing Eff: ",grid.score(Xtest,ytest))

In [ ]:
print("R2: ",r2_score(ytest,ypred))
print("MAE: ",mean_absolute_error(ytest,ypred))

In [ ]:
df.to_csv("cleaned_df.csv")


In [ ]:
joblib.dump(best_model,'rf_model.joblib')

In [ ]:
joblib.dump(Xtrain.columns.tolist(),"model_columns.joblib")